# FreshMart Product Data — Exploratory Analysis

This notebook reproduces the original EDA on the FreshMart product catalogue (205 products, 6 columns) using the restructured project layout.

**Data source:** `data/raw/freshmart_products_csv.csv`  
**Cleaned pipeline:** see `scripts/run_pipeline.py` and `src/` for the reusable modules  
**Database:** the project defaults to SQLite (no server needed); see README for PostgreSQL setup

In [ ]:
import pandas as pd
from pathlib import Path

# Use the CSV from the project's data/raw/ directory
csv_path = Path("data/raw/freshmart_products_csv.csv")
df = pd.read_csv(csv_path)
print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
df.head()

In [ ]:
# Missing values per column
df.isnull().sum()

In [ ]:
# Data types and non-null counts
df.info()

In [ ]:
# Derive StockValue = Price * StockQuantity
df["StockValue"] = df["Price"] * df["StockQuantity"]
df.head()

In [ ]:
# Average price
df["Price"].mean()

In [ ]:
# Total stock quantity
df["StockQuantity"].sum()

In [ ]:
# Category breakdown — note misspellings/variants in the raw data
print("Raw categories (before normalisation):")
print(df["Category"].value_counts())
print()
print("Unique raw categories:", sorted(df["Category"].unique()))

## Data quality note

The raw CSV contains three category variants that should be merged into their canonical forms:

| Raw value | Canonical | Issue |
|-----------|-----------|-------|
| `dairy` | `Dairy` | lowercase |
| `Bevarages` | `Beverages` | typo |
| `snakcs` | `Snacks` | typo |

These are handled automatically by `src/data_loader.clean()` via the `CATEGORY_NORMALISATION` mapping in `src/config.py`. After cleaning, the dataset has 5 distinct categories.

In [ ]:
# Use the project's cleaning function to see the normalised result
import sys
sys.path.insert(0, ".")

from src.data_loader import clean
df_clean = clean(df.copy())

print("Categories after cleaning:")
print(df_clean["Category"].value_counts())
print()
print("Number of distinct categories:", df_clean["Category"].nunique())

## Database

The original notebook used a local PostgreSQL connection with hardcoded credentials. The restructured project replaces this with:

- `src/database.py` — driver-agnostic layer (SQLite by default, PostgreSQL optional)
- `scripts/run_pipeline.py` — end-to-end pipeline that loads, cleans, analyses, and persists to SQLite

No credentials are needed for the default SQLite path. See `.env.example` and the README for the PostgreSQL option.